# espeak-ng

A refresher on **espeak-ng** — a tiny, fully-offline **formant** speech synthesizer and, more importantly today, the de-facto **grapheme-to-phoneme (G2P) front end** for modern neural TTS. It turns text into **IPA phonemes** using a dictionary + letter-to-sound rules for **100+ languages**, and can also speak that text with a compact, unmistakably robotic formant voice. Piper, Coqui, Bark, StyleTTS2 and Tacotron-style models almost all phonemize their input with espeak-ng (usually via the `phonemizer` package); the synthesizer itself lives on in accessibility tools where size and determinism beat naturalness.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _the phoneme-stream parser (Example 1) and a from-scratch formant vowel (Example 2) run on stdlib + numpy with no install; the real `espeak-ng` CLI/`phonemizer` call (Example 3) is gated behind `shutil.which` + `RUN_ESPEAK`_

## 1. What & Why

**What it is.** `espeak-ng` (the maintained fork of Jonathan Duddington's original *eSpeak*) is a **formant synthesizer**: it generates speech directly from acoustic rules — formant frequencies, voicing, durations — rather than from recorded audio or a neural network. The whole engine plus voice data is a few **megabytes**, needs no GPU, no model download, and is deterministic. It ships as a CLI (`espeak-ng`), a C library (`libespeak-ng`), and is wrapped by Python via `phonemizer` (the `espeak` backend) or `py-espeak-ng`.

**The two problems it solves.** (1) **Text → phonemes (G2P).** Given text in any of 100+ languages, espeak-ng produces **IPA** (or Kirshenbaum / its own notation) phoneme strings using a pronunciation dictionary plus letter-to-sound rules. This is the part everyone uses today: nearly every open neural TTS — Piper, Coqui (VITS/XTTS), Bark, StyleTTS2, Tacotron 2 — feeds on espeak-ng phonemes, normally through the `phonemizer` library. (2) **Actual speech synthesis**, when you need *some* voice with a near-zero footprint and total offline determinism: screen readers, embedded/IoT devices, low-vision accessibility, quick "does this text parse?" checks.

**When to reach for it.** You need a **phonemizer** for a TTS or pronunciation pipeline; you want fully **offline, tiny, deterministic** speech (accessibility, embedded, kiosks with no network/GPU); you need **broad language coverage** fast; you want a no-dependency way to inspect how text is pronounced.

**When not to.** You want **natural-sounding** speech — formant synthesis is robotic and obviously artificial; use Piper for offline-but-natural, or a cloud voice / ElevenLabs / StyleTTS2 for high naturalness. espeak-ng's value today is overwhelmingly as the **phoneme front end** *to* those systems, not as the voice your users hear.

## 2. Mental Model

**espeak-ng is a rules engine, not a recording and not a network. Text walks through a dictionary + letter-to-sound rules to become a phoneme string; that phoneme string is then either handed to a neural TTS (the modern use) or run through a formant synthesizer that fakes a voice with a handful of resonators (the classic use). There is nothing learned and nothing stored as audio — same input, same output, every time.**

```
                                         ┌─────────────── modern use ───────────────┐
                                         │  phonemes ─▶ phonemizer ─▶ neural TTS     │
  text          dictionary + L2S rules   │            (Piper/Coqui/Bark/StyleTTS2)  │
 ┌────────┐   per-language pronunciation  │                                          │
 │"Hello."│ ─▶ h ə l ˈoʊ  ───────────────▶┤                                          │
 └────────┘   (IPA / Kirshenbaum)         │  ┌──────────── classic use ───────────┐ │
                                         └──│ phonemes ─▶ FORMANT synth ─▶ wav    │ │
                                            │  glottal source + F1/F2/F3 filters  │ │
                                            │  (tiny, deterministic, robotic)     │ │
                                            └─────────────────────────────────────┘
```

Three ideas make it click:

1. **Phonemes are the product, audio is optional.** The high-value output is the *phoneme string*. `espeak-ng -x` / `--ipa -q` gives you phonemes **without** ever synthesizing audio — that `-q` (quiet) path is exactly what `phonemizer` calls under the hood.
2. **Formant synthesis = source–filter, by hand.** A voiced sound is a buzzy glottal **source** (a pulse train at the pitch) pushed through a few **formant** resonators (F1, F2, F3…) that shape it into a vowel. espeak-ng computes these from rules per phoneme. That's why it's small and robotic — there's no real vocal tract, just a handful of tuned band-pass filters.
3. **It's a front end, not a destination.** In 2025 you almost never *ship* espeak-ng's voice; you ship its **phonemes** into a better synthesizer. Knowing espeak-ng means knowing the G2P layer the whole open-TTS stack stands on.

## 3. Key Concepts

- **G2P / phonemization.** Grapheme-to-phoneme: text → phoneme symbols. espeak-ng does it with a per-language **pronunciation dictionary** (`*_dict`) plus **letter-to-sound rules** for out-of-dictionary words. This is its dominant role today.
- **Formant synthesis.** Speech generated from a **glottal source** (pulse train for voiced sounds, noise for fricatives) filtered by **formant** resonances (F1, F2, F3…). No recordings, no neural net — hence tiny, deterministic, and robotic. Contrast with **concatenative** (stitches recorded units) and **neural** (Piper/VITS) synthesis.
- **Phoneme notations.** Output can be **IPA** (`--ipa`), **Kirshenbaum** ASCII-IPA (`-x`), or espeak's internal notation. `phonemizer` typically requests IPA. **Stress** marks (`ˈ` primary, `ˌ` secondary) and **language switches** ride along in the stream.
- **Voices & languages.** A *voice* is a language/accent config (`-v en-us`, `-v fr`, `-v cmn`); `espeak-ng --voices` lists 100+. "MBROLA" voices add diphone data for less-robotic output but need extra packages.
- **Prosody knobs.** `-s` words-per-minute (default ~175), `-p` pitch (0–99), `-a` amplitude, `-g` word gap. Plus limited **SSML** (`<break>`, `<emphasis>`, `<prosody>`, `<say-as>`).
- **The `phonemizer` package.** The standard Python bridge: `phonemizer.backend.EspeakBackend("en-us")` shells out to `libespeak-ng`. **`with_stress=True`** keeps stress marks; **`preserve_punctuation`** keeps punctuation. This is how neural-TTS training pipelines get their phonemes.
- **Output / audio.** `espeak-ng -w out.wav "..."` writes a **WAV**; `--stdout` pipes raw audio; default speaks to the sound card. Sample rate is **22050 Hz** mono for the formant voice.
- **Determinism & footprint.** Same text + same voice + same flags → byte-identical phonemes (and near-identical audio). The engine + data are a few MB and run anywhere C runs — the reason it's embedded in screen readers (NVDA, Orca) and edge devices.

## 4. Setup

```bash
# System package (the CLI + libespeak-ng the Python wrappers call):
sudo apt-get install espeak-ng          # Debian/Ubuntu
brew install espeak-ng                   # macOS
# Windows: installer from the espeak-ng GitHub releases

# Python G2P bridge (what neural-TTS pipelines actually use):
pip install phonemizer                   # imports as `phonemizer`, needs libespeak-ng present
```

Quick CLI smoke tests:

```bash
espeak-ng "Hello from espeak."                 # speak to the sound card
espeak-ng --ipa -q "Hello from espeak."        # IPA phonemes only, no audio   -> hɛlˈoʊ fɹˈʌm ɛspˈiːk
espeak-ng -x  -q "Hello from espeak."          # Kirshenbaum ASCII phonemes, no audio
espeak-ng -v fr "Bonjour le monde."            # French voice
espeak-ng -w out.wav "Saved to a file."        # write a WAV instead of speaking
espeak-ng --voices                              # list all languages/voices
```

> **Two things to know.** (1) On macOS, Homebrew installs the library but `phonemizer` sometimes
> can't find it — set `PHONEMIZER_ESPEAK_LIBRARY=/opt/homebrew/lib/libespeak-ng.dylib`.
> (2) This notebook **doesn't require espeak-ng to run**: Examples 1–2 reproduce the phoneme
> stream and the formant-synthesis idea with only the standard library + `numpy`. Example 3 runs
> the *real* `espeak-ng` CLI and the `phonemizer` API, gated behind `shutil.which` + `RUN_ESPEAK`,
> so the notebook executes top-to-bottom either way.

In [ ]:
import shutil, sys
import numpy as np

ESPEAK = shutil.which("espeak-ng") or shutil.which("espeak")
print(f"python {sys.version.split()[0]}, numpy {np.__version__}")
print(f"espeak-ng on PATH: {ESPEAK or 'NOT FOUND (Examples 1-2 still run; Example 3 is gated)'}")
print("Pipeline: text -> [dictionary + letter-to-sound rules] -> phonemes -> {neural TTS | formant synth -> wav}")

## 5. Worked Examples

### Example 1 — Reading espeak-ng's phoneme stream (the part neural TTS consumes)

The output you actually feed downstream is the **phoneme string**. Running
`espeak-ng --ipa -q "Hello world. It costs $5."` yields IPA like
`hɛlˈoʊ wˈɜːld. ɪt kˈɒsts fˈaɪv dˈɒlɚz`. Two things matter to a TTS: the **stress marks**
(`ˈ` primary, `ˌ` secondary) attached to the following vowel, and that espeak has already
**expanded `$5` to "five dollars"** via its text normalizer. Below we hard-code that exact
espeak output and parse it the way a phonemizer-fed pipeline does — split into words, strip and
record stress, and confirm the normalization happened. No install required.

In [ ]:
# Hard-coded output of:  espeak-ng --ipa -q "Hello world. It costs $5."
espeak_ipa = "hɛlˈoʊ wˈɜːld. ɪt kˈɒsts fˈaɪv dˈɒlɚz"

PRIMARY, SECONDARY = "ˈ", "ˌ"

def analyze(word):
    """Record which stresses a word carries, then return the bare phoneme symbols."""
    stresses = []
    if PRIMARY in word:   stresses.append("primary")
    if SECONDARY in word: stresses.append("secondary")
    bare = word.replace(PRIMARY, "").replace(SECONDARY, "")
    return bare, (stresses or ["none"])

words = espeak_ipa.split()
print(f"raw IPA from espeak : {espeak_ipa}\n")
for w in words:
    bare, stresses = analyze(w)
    print(f"  {w:<9} -> phonemes {bare:<8} | stress: {', '.join(stresses)}")

# espeak's text normalizer turned "$5" into spoken words BEFORE phonemizing:
normalized = "five dollars" if "fˈaɪv" in espeak_ipa and "dˈɒlɚz" in espeak_ipa else "?"
print(f"\ntext-normalization: '$5'  ->  '{normalized}'  (done by espeak, not by you)")
print(f"phoneme symbols (deduped): {sorted({c for w in words for c in analyze(w)[0] if c.isalpha() or c in 'ɜɒɛɚaɪoʊ'})}")

### Example 2 — Formant synthesis from scratch: why espeak sounds robotic

espeak's *voice* is a **source–filter** model. For a voiced sound it makes a buzzy **glottal
source** — a pulse train at the pitch (F0) — and pushes it through a few **formant** resonators
(F1, F2, F3) that sculpt it into a particular vowel. Below we synthesize the vowel **/ɑ/**
("ah") exactly that way with `numpy`: build a pulse train at 120 Hz, then apply three resonant
band-pass filters at the textbook formants for /ɑ/, and write a real WAV with the stdlib `wave`
module at espeak's 22050 Hz. The result *is* recognizably a vowel — and recognizably synthetic,
because there's no real vocal tract, just three tuned resonators.

In [ ]:
import wave

SAMPLE_RATE = 22050          # espeak-ng's formant voice sample rate
F0 = 120.0                   # glottal pitch (Hz) -> a low male-ish voice
DUR = 0.6
N = int(SAMPLE_RATE * DUR)
t = np.arange(N) / SAMPLE_RATE

# 1) Glottal SOURCE: a pulse train at F0 (one impulse per pitch period).
period = int(SAMPLE_RATE / F0)
source = np.zeros(N, dtype=np.float32)
source[::period] = 1.0

# 2) FILTER: three resonators at the formants of /ɑ/ ("ah"). Each is a 2-pole
#    resonator y[n] = x[n] + 2r·cos(w)·y[n-1] - r²·y[n-2]  (a band-pass peak).
def resonator(x, freq, bw=90.0):
    r = np.exp(-np.pi * bw / SAMPLE_RATE)
    w = 2 * np.pi * freq / SAMPLE_RATE
    a1, a2 = 2 * r * np.cos(w), -(r * r)
    y = np.zeros_like(x)
    for n in range(len(x)):
        y[n] = x[n] + a1 * (y[n-1] if n >= 1 else 0) + a2 * (y[n-2] if n >= 2 else 0)
    return y

formants = [(700, 130), (1100, 90), (2600, 120)]   # (F1, F2, F3) for /ɑ/
voiced = np.zeros(N, dtype=np.float32)
for f, bw in formants:
    voiced += resonator(source, f, bw)

voiced *= np.hanning(N).astype(np.float32)          # fade in/out
voiced /= np.abs(voiced).max()                       # normalize to [-1, 1]
pcm16 = (np.clip(voiced, -1, 1) * 32767).astype(np.int16)

out = "/tmp/espeak_formant_ah.wav"
with wave.open(out, "wb") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(SAMPLE_RATE)
    w.writeframes(pcm16.tobytes())

print(f"synthesized /ɑ/ via source-filter formant synthesis -> {out}")
print(f"  pitch F0={F0:.0f} Hz, formants F1/F2/F3={[f for f,_ in formants]} Hz")
print(f"  {SAMPLE_RATE} Hz mono 16-bit, {N} samples ({DUR:.1f}s), peak {np.abs(pcm16).max()}/32767")
print("This is the entire idea behind espeak's voice: a buzz shaped by a few resonators -> robotic but intelligible.")

### Example 3 — The real engine: `espeak-ng` CLI + the `phonemizer` API (gated)

With `espeak-ng` installed, phonemizing and synthesizing are one-liners. This is gated behind
`shutil.which("espeak-ng")` **and** `RUN_ESPEAK=1` so the notebook still runs without it. Either
way the cell prints the canonical shapes: the CLI for phonemes/WAV, and the `phonemizer`
`EspeakBackend` call that neural-TTS training pipelines use to convert a whole corpus to
phonemes.

In [ ]:
import os, subprocess

text = "Hello world. It costs five dollars."

if ESPEAK and os.getenv("RUN_ESPEAK"):
    # IPA phonemes only (-q = no audio): exactly what phonemizer shells out to do.
    ipa = subprocess.run([ESPEAK, "--ipa", "-q", text],
                         capture_output=True, text=True).stdout.strip()
    print(f"espeak IPA : {ipa}")
    # Write a real WAV with the formant voice.
    subprocess.run([ESPEAK, "-w", "/tmp/espeak_real.wav", text], check=True)
    print("wrote /tmp/espeak_real.wav")
    try:
        from phonemizer import phonemize
        py_ipa = phonemize(text, language="en-us", backend="espeak", with_stress=True)
        print(f"phonemizer : {py_ipa}")
    except ImportError:
        print("(pip install phonemizer for the Python G2P bridge)")
else:
    print("espeak-ng not available / RUN_ESPEAK unset — showing canonical call shapes:\n")
    print("# CLI: phonemes only (no audio) — this is what phonemizer calls internally")
    print(f'espeak-ng --ipa -q "{text}"      # -> hɛlˈoʊ wˈɜːld. ɪt kˈɒsts fˈaɪv dˈɒlɚz')
    print(f'espeak-ng -x   -q "{text}"      # Kirshenbaum ASCII phonemes instead of IPA')
    print(f'espeak-ng -v fr -s 150 "Bonjour"  # French voice, 150 wpm')
    print(f'espeak-ng -w out.wav "{text}"    # write a WAV with the formant voice\n')
    print("# Python G2P bridge used by Piper / Coqui / StyleTTS2 training pipelines:")
    print("from phonemizer.backend import EspeakBackend")
    print('backend = EspeakBackend("en-us", with_stress=True, preserve_punctuation=True)')
    print('phonemes = backend.phonemize(["Hello world."])   # -> ["həlˈoʊ wˈɜːld."]')

## 6. Gotchas & Pitfalls

- **You almost always want the phonemes, not the voice.** New users synthesize WAVs and conclude
  "espeak sounds terrible." That's not the point in 2025 — its job is the **G2P front end** for a
  *real* TTS. Reach for `--ipa -q` (phonemes, no audio), not the formant voice.
- **`phonemizer` can't find `libespeak-ng`.** Especially on macOS/Homebrew. Set
  `PHONEMIZER_ESPEAK_LIBRARY=/opt/homebrew/lib/libespeak-ng.dylib` (or the Linux `.so` path).
  Missing-library errors look like crashes, not "not installed."
- **`espeak` vs `espeak-ng`.** Old guides/binaries call `espeak`; the maintained engine is
  `espeak-ng`. Different phoneme output and flags in places — pin to `espeak-ng`.
- **Stress marks toggle silently.** `phonemizer` defaults to **dropping** stress; pass
  `with_stress=True` or your downstream model loses the `ˈ`/`ˌ` it was trained on. Likewise
  `preserve_punctuation=True` to keep `.`/`,` that cue prosody.
- **Hidden text normalization.** espeak expands numbers, currency, dates, and abbreviations
  ("$5" → "five dollars", "Dr." → "doctor") *before* phonemizing — language-specific and not
  always what you want. Inspect the phonemes; normalize upstream if you need control.
- **Language switching injects markers.** Mixed-language text can emit `(en)`/`(fr)` language-
  switch tokens in the phoneme stream. Strip or handle them, or they pollute your phoneme set.
- **Mismatched phonemizer between training and inference = garbage.** If a model was trained on
  espeak-ng IPA, you must phonemize inference text the **same way** (same version, flags,
  language). A different G2P (or a different espeak version) shifts the symbol set and the model
  mispronounces everything.
- **It's robotic by design.** Formant synthesis will never sound natural; MBROLA voices help a
  little but add data and licensing friction. Don't fight it — if you need natural offline audio,
  that's Piper's job.
- **Default speech rate is fast.** ~175 wpm out of the box; set `-s 130`–`150` for clearer audio
  in accessibility use.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs espeak-ng |
|---|---|---|
| **espeak-ng (as G2P)** | The **phonemizer** front end for any neural TTS; broad language coverage; tiny, deterministic | Phonemes only — you still need a synthesizer for natural audio |
| **espeak-ng (as voice)** | Ultra-light, fully offline, deterministic speech; screen readers, embedded, accessibility | Robotic/formant — obviously synthetic; no naturalness |
| **`gruut` / `g2p_en` / `phonemizer`+other backends** | Alternative G2P (gruut is pure-Python, no system dep) | Fewer languages or different symbol set; espeak-ng is the most widely-trained-against |
| **Piper** | Offline, **natural-ish**, real-time TTS on CPU/Pi — *uses espeak-ng for G2P internally* | Heavier (ONNX + voice file); not a phonemizer you call standalone |
| **Coqui / StyleTTS2 / Bark** | Self-hosted high-quality / expressive / cloning neural TTS — *also phonemize via espeak-ng* | GPU-hungry, large; espeak-ng is just their input layer |
| **Cloud (Azure / Google / Polly / ElevenLabs)** | Best naturalness, rich SSML, zero setup | Paid, online, audio leaves your box; do their own G2P |

**Rule of thumb:** in a modern stack, reach for **espeak-ng as the G2P layer** — it's the phoneme
front end Piper, Coqui, StyleTTS2 and Bark all stand on, and `phonemizer` is how you call it.
Use its **formant voice** only when you need *some* intelligible speech at near-zero footprint
with total offline determinism (accessibility, embedded). For natural offline audio step up to
**Piper**; for cloning/expressiveness to **Coqui/StyleTTS2**; for the highest naturalness with
the least ops, a **cloud voice**.

## 8. Resources

- **espeak-ng — GitHub (engine, voices, docs):** https://github.com/espeak-ng/espeak-ng
- **Command-line options & usage guide:** https://github.com/espeak-ng/espeak-ng/blob/master/docs/guide.md
- **Adding/understanding phoneme & dictionary rules:** https://github.com/espeak-ng/espeak-ng/blob/master/docs/dictionary.md
- **`phonemizer` — the Python G2P bridge used by neural TTS:** https://github.com/bootphon/phonemizer
- **List of supported languages/voices:** https://github.com/espeak-ng/espeak-ng/blob/master/docs/languages.md
- **Formant / source–filter synthesis background (Klatt synthesizer):** https://en.wikipedia.org/wiki/Formant_synthesis
- **Piper — a neural TTS that uses espeak-ng for phonemization:** https://github.com/OHF-Voice/piper1-gpl

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
PRIMARY, SECONDARY = "\u02c8", "\u02cc"


def parse_phonemes(stream, with_stress=True, preserve_punctuation=True):
    ...


def resonator(source, freq, sample_rate, bandwidth=90.0):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE